# Creating HF Dataset for Mistral Fine-tuning

## Install Dependencies

In [ ]:
# Run this cell first — installs everything needed
!pip install -q datasets huggingface_hub

## Mount Google Drive
This saves the dataset to your Drive so it persists after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# This folder will be created automatically if it doesn't exist
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/AamirGPT'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f'Project folder ready: {DRIVE_PROJECT_DIR}')

## Imports

In [ ]:
import csv
import random
from datasets import Dataset, DatasetDict

## Prep Training Examples

In [ ]:
# load csv of YouTube comments
# The CSV file is included in the repo at data/YT-comments.csv
comment_list = []
response_list = []

with open('data/YT-comments.csv', mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)
    for line in reader:
        if line[0] == 'Comment':
            continue
        comment_list.append(line[0])
        response_list.append(line[1] + " -AamirGPT")

print(f'Loaded {len(comment_list)} comments')

In [ ]:
# Build prompt format for Mistral instruction tuning
instructions_string = """AamirGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. \
It reacts to feedback aptly and ends responses with its signature '–AamirGPT'. \
AamirGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, \
thus keeping the interaction natural and engaging.

Please respond to the following comment.
"""

example_template = lambda comment, response: f'''<s>[INST] {instructions_string} \n{comment} \n[/INST]\n''' + response + "</s>"

example_list = []
for i in range(len(comment_list)):
    example = example_template(comment_list[i], response_list[i])
    example_list.append(example)

print(f'Created {len(example_list)} examples')
print('\nSample example:')
print(example_list[0])

In [ ]:
# create train/test split (9 examples for test)
random.seed(42)  # fixed seed for reproducibility
test_index_list = random.sample(range(0, len(example_list) - 1), 9)

test_list = [example_list[index] for index in test_index_list]
for example in test_list:
    example_list.remove(example)

print(f'Train examples: {len(example_list)}')
print(f'Test examples:  {len(test_list)}')

## Create HuggingFace Dataset

In [ ]:
data = DatasetDict({
    'train': Dataset.from_dict({"example": example_list}),
    'test':  Dataset.from_dict({"example": test_list})
})
print(data)

## Save Dataset to Google Drive

In [ ]:
# Save to Google Drive so it survives session resets
dataset_save_path = f'{DRIVE_PROJECT_DIR}/data'
data.save_to_disk(dataset_save_path)
print(f'Dataset saved to: {dataset_save_path}')

## (Optional) Push Dataset to HuggingFace Hub
Only needed if you want the dataset publicly available on HuggingFace.

In [ ]:
# Login with your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
from huggingface_hub import login

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxx"  # <-- paste your HF write token here
login(HF_TOKEN)

In [ ]:
HF_USERNAME = "your-hf-username"  # <-- replace with your HuggingFace username
data.push_to_hub(f"{HF_USERNAME}/aamir-youtube-comments")
print('Dataset pushed to HuggingFace Hub!')